In [0]:
# ============================================================
# Step 1 - Load Data
# ============================================================

from pyspark.sql.functions import col, year, month

# Read transaction CSV file
transactions_df = (
    spark.read.format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .option("path","/Volumes/workspace/default/source_data/transactions.csv")
    .load()
)

display(transactions_df)


In [0]:
# ============================================================
# Step 2 - Extract Year & Month
# ============================================================

transactions_df = (
    transactions_df
    .withColumn("year",year(col("sale_date")))
    .withColumn("month",month(col("sale_date")))
)

display(transactions_df)

In [0]:
# ============================================================
# Step 3 - Aggregate Monthly Sales
# ============================================================

from pyspark.sql.functions import sum

# Calculate total quantity per store, per product, per year, per month
monthly_sales_df = (
    transactions_df
        .groupBy("store_id","product_id","year","month")
        .agg(
            sum("quantity").alias("monthly_quantity")
        )
        .orderBy("store_id","product_id","year","month")
)
display(monthly_sales_df)

In [0]:
# ============================================================
# Step 4 - Create a Previous Month Feature
# ============================================================

from pyspark.sql.window import Window
from pyspark.sql.functions import lag

# Define a Window per Store & Product
window_spec = (
    Window
        .partitionBy("store_id","product_id")
        .orderBy("year","month")
)

# Create previous month quantity column
monthly_sales = (
    monthly_sales_df
        .withColumn("prev_month_quantity",lag("monthly_quantity",1).over(window_spec))
        .orderBy("store_id","product_id","year","month")
)
monthly_sales.display()

In [0]:
# ============================================================
# Step 5 - Prepare ML Dataset
# ============================================================

# Rename montly quantity to 'label'
# MLlib expects target column to be named 'label'
gold_df = monthly_sales.withColumnRenamed("monthly_quantity","label")
display(gold_df)

In [0]:
# ============================================================
# Step 6 - Convert Feature to Vector
# ============================================================
from pyspark.ml.feature import VectorAssembler

# Make sure no null values in Feature & label Column
gold_df = gold_df.dropna(subset=["prev_month_quantity","label"])
print("gold_df.count():",gold_df.count())

# MLlib requires features in vector format
assembler = VectorAssembler(
    inputCols=["prev_month_quantity"], # Our only feature
    outputCol="features"
)

model_df = assembler.transform(gold_df)
display(model_df.select("prev_month_quantity", "features","label"))



In [0]:
# ============================================================
# Step 7 - Train/Test Split
# ============================================================

# Split data into training (80%) and testing (20%)
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)

display(train_df)
display(test_df)

print(f"Training Dataset Count: {train_df.count()}")
print(f"Testing Dataset Count: {test_df.count()}")

In [0]:
# ============================================================
# Step 8 - Train Linear Regression Model
# ============================================================

from pyspark.ml.regression import LinearRegression

# Create Linear Regression Model
lr = LinearRegression(
    featuresCol="features",
    labelCol="label"
)

# Train the model
model = lr.fit(train_df)

In [0]:
# ============================================================
# Step 9 - Make Predictions
# ============================================================

predications = model.transform(test_df)
display(predications.select("store_id","product_id","label","prev_month_quantity","prediction"))

In [0]:
# ============================================================
# Step 10 - Evaluate Model
# ============================================================

from pyspark.ml.evaluation import RegressionEvaluator

# Evaluate the model using Root Mean Squared Error
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator.evaluate(predications)
print(f"RMSE: {rmse}")